### Installation

In [ ]:

import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==5.3.0
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel 
import torch, os, json, statistics

# ── Kaggle paths ──────────────────────────────────────────────────────────────
TRAIN_PATH = "/kaggle/input/datasets/sathwikskdfj/resumedataset/train_clean.jsonl"
VAL_PATH   = "/kaggle/input/datasets/sathwikskdfj/resumedataset/val.jsonl"
OUTPUT_DIR = "/kaggle/working/qwen35-checkpoints"
LORA_DIR   = "/kaggle/working/qwen35-lora"
GGUF_DIR   = "/kaggle/working/qwen35-gguf"
for d in [OUTPUT_DIR, LORA_DIR, GGUF_DIR]:
    os.makedirs(d, exist_ok=True)

INFERENCE_SYSTEM_PROMPT = (
    "You are an expert technical recruiter specializing in software engineering roles. "
    "Given a job description and a list of candidate resumes, rank all candidates from best to worst fit. "
    "Return ONLY a valid JSON object - no markdown, no explanation, no extra text."
)

# ── Load model ────────────────────────────────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained( # CHANGED
    "unsloth/Qwen3.5-4B",
    load_in_4bit  = False,        
    load_in_16bit = True,         
    use_gradient_checkpointing = "unsloth",
)

We now add LoRA adapters for parameter efficient finetuning.


In [ ]:
model = FastLanguageModel.get_peft_model( # CHANGED
    model,
    r = 16,           
    lora_alpha = 16,  
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  
    loftq_config = None, 
    target_modules = [   # ADDED standard text target modules
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

<a name="Data"></a>
### Data Prep
We use the **ResumeDataset** (add via Kaggle → Add Data → Your Datasets → ResumeDataset).

Each JSONL line has a `messages` key with 3 turns: system, user (job description + resumes), and assistant (JSON ranking). We validate the dataset first, then load it for training.

In [ ]:
from datasets import load_dataset

# ── Validate dataset ──────────────────────────────────────────────────────────
def audit_dataset(path, label):
    errors, scores_all, domain_counts = [], [], {}
    with open(path) as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        try:
            sample = json.loads(line)
            msgs   = sample.get("messages", [])
            if len(msgs) != 3:
                errors.append(f"Line {i+1}: {len(msgs)} messages (expected 3)")
                continue
            output = json.loads(msgs[2]["content"])
            if not all(k in output for k in ["ranking", "scores", "reasons"]):
                errors.append(f"Line {i+1}: missing keys in assistant turn")
            scores_all.extend(output["scores"].values())
            domain = sample.get("metadata", {}).get("domain", "unknown")
            domain_counts[domain] = domain_counts.get(domain, 0) + 1
        except Exception as e:
            errors.append(f"Line {i+1}: {e}")
    print(f"\n{'='*40}\n{label}: {len(lines)} samples, {len(errors)} errors")
    for e in errors[:3]: print(f"  {e}")
    if scores_all:
        m, s = statistics.mean(scores_all), statistics.stdev(scores_all)
        print(f"{'OK' if 55<=m<=65 else 'WARN'} Mean: {m:.2f}  {'OK' if s>18 else 'WARN'} Std: {s:.2f}")
    print("Domains:", dict(sorted(domain_counts.items(), key=lambda x: -x[1])))

if os.path.exists(TRAIN_PATH):
    audit_dataset(TRAIN_PATH, "TRAIN")
    audit_dataset(VAL_PATH,   "VAL")
else:
    print("WARNING: Dataset not found. Check Kaggle → Add Data → ResumeDataset.")

# ── Load datasets ─────────────────────────────────────────────────────────────
train_dataset = load_dataset("json", data_files=TRAIN_PATH, split="train")
val_dataset   = load_dataset("json", data_files=VAL_PATH,   split="train")

In [ ]:
# Preview dataset sample
print("Train samples:", len(train_dataset))
print("Val   samples:", len(val_dataset))
print("\nSample messages structure:")
sample = train_dataset[0]
for msg in sample["messages"]:
    role = msg["role"]
    content_preview = msg["content"][:120] + "..." if len(msg["content"]) > 120 else msg["content"]
    print(f"  [{role}]: {content_preview}")

In [ ]:
# Pre-training inference: see what the base model outputs before fine-tuning
FastLanguageModel.for_inference(model)

TEST_INPUT = """JOB DESCRIPTION:
Company: Nexus Systems
Role: Mid-Level Backend Engineer
Experience Required: 3-5 years
Core Requirements: Python, FastAPI, PostgreSQL, Redis.
Nice-to-Have: Experience with Rust for performance-critical microservices, and a strong grasp of multithreading or concurrent systems.

RESUMES:
Candidate A: Sarah Chen. 4 years of experience. Previously at Stripe. Designed and maintained high-traffic REST APIs using Python and FastAPI. Managed database migrations in PostgreSQL and implemented Redis caching layers. Recently rewrote a bottlenecked data processing module in Rust to improve thread safety and execution speed.

Candidate B: James Wilson. 2 years of experience. Junior web developer. Primarily worked with Node.js and basic Django. Wrote simple SQL queries for internal dashboards. No direct experience with FastAPI, Redis, or high-concurrency architectures.

Candidate C: Dr. Maria Santos. 6 years of experience. Academic researcher transitioning to industry. Built complex machine learning models and data pipelines using Python (Pandas, PyTorch), MATLAB, and R. Highly analytical, but lacks production backend engineering or web framework experience."""

messages = [
    {"role": "system", "content": INFERENCE_SYSTEM_PROMPT},
    {"role": "user",   "content": TEST_INPUT},
]

# Text-only input — no image argument
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text=input_text, add_special_tokens=False, return_tensors="pt").to("cuda") #

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
print("=== Base model output (before fine-tuning) ===")
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=256,
                   use_cache=True, temperature=0.1, min_p=0.1)

<a name="Train"></a>
### Train the model


In [ ]:
def format_and_tokenize(examples):
    # 1. Apply the chat template
    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) 
        for msg in examples["messages"]
    ]
    # 2. Add padding=True to fix the inhomogeneous shape error
    return tokenizer(text=texts, truncation=True, max_length=2048, padding=True)

# Run the mapping
train_dataset = train_dataset.map(
    format_and_tokenize, 
    batched=True,
    remove_columns=train_dataset.column_names
)

In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model) 

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,   
        gradient_accumulation_steps = 16, # Total effective batch of 16
        warmup_steps = 2,
        num_train_epochs = 1,
        learning_rate = 5e-5,
        logging_steps = 1,
        packing = True,          # Enable this for major speedup
        optim = "adamw_8bit",
        weight_decay = 0.01,     # Slight increase for better regularization in 2 epochs
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",     
        max_seq_length = 4096,   # MUST be 'max_seq_length' for SFTConfig
        dataset_text_field = "text", # Required if using packing=True
        save_steps = 20
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

In [ ]:
# Plot loss curve
import matplotlib.pyplot as plt

state_path = f"{OUTPUT_DIR}/trainer_state.json"
if os.path.exists(state_path):
    with open(state_path) as f:
        history = json.load(f)["log_history"]
    train_log = [(h["step"], h["loss"])      for h in history if "loss" in h and "eval_loss" not in h]
    eval_log  = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]
    if train_log and eval_log:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot([s for s,_ in train_log], [l for _,l in train_log], label="Train", alpha=0.7)
        ax.plot([s for s,_ in eval_log],  [l for _,l in eval_log],  label="Val",   marker="o")
        ax.set(xlabel="Step", ylabel="Loss", title="Qwen 3.5 — Loss Curve")
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig("/kaggle/working/loss_curve.png", dpi=150)
        plt.show()
else:
    print("trainer_state.json not found — training may not have completed.")

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
# Post-training inference — text-only resume ranking
FastLanguageModel.for_inference(model)

TEST_INPUT = """JOB DESCRIPTION:
Company: Nexus Systems
Role: Mid-Level Backend Engineer
Experience Required: 3-5 years
Core Requirements: Python, FastAPI, PostgreSQL, Redis.
Nice-to-Have: Experience with Rust for performance-critical microservices, and a strong grasp of multithreading or concurrent systems.

RESUMES:
Candidate A: Sarah Chen. 4 years of experience. Previously at Stripe. Designed and maintained high-traffic REST APIs using Python and FastAPI. Managed database migrations in PostgreSQL and implemented Redis caching layers. Recently rewrote a bottlenecked data processing module in Rust to improve thread safety and execution speed.

Candidate B: James Wilson. 2 years of experience. Junior web developer. Primarily worked with Node.js and basic Django. Wrote simple SQL queries for internal dashboards. No direct experience with FastAPI, Redis, or high-concurrency architectures.

Candidate C: Dr. Maria Santos. 6 years of experience. Academic researcher transitioning to industry. Built complex machine learning models and data pipelines using Python (Pandas, PyTorch), MATLAB, and R. Highly analytical, but lacks production backend engineering or web framework experience."""

messages = [
    {"role": "system", "content": INFERENCE_SYSTEM_PROMPT},
    {"role": "user",   "content": TEST_INPUT},
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text=input_text, add_special_tokens=False, return_tensors="pt").to("cuda") #


with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens     = 256,
        temperature        = 0.1,
        min_p              = 0.1,
        repetition_penalty = 1.5,
        do_sample          = True,
        use_cache          = True,
    )
raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=== Fine-tuned model output ===")
print(raw)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
# Save LoRA adapters to Kaggle working directory
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"LoRA saved to: {LORA_DIR}")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN")

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Export to GGUF (q5_k_m recommended — good size/quality tradeoff)
# NOTE: vLLM 0.16.0 does NOT support Qwen3.5 — GGUF/llama.cpp is the correct export path
print("Exporting GGUF Q5_K_M — takes 10-20 min, do not interrupt...")
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q5_k_m")

import shutil
gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf")]
for f in gguf_files:
    fpath = os.path.join(GGUF_DIR, f)
    print(f"{f}: {os.path.getsize(fpath)/1e9:.2f} GB")

# Copy GGUF + zip LoRA for easy Kaggle output download
if gguf_files:
    shutil.copy(os.path.join(GGUF_DIR, gguf_files[0]), "/kaggle/working/qwen35_finetuned_Q5_K_M.gguf")
    print("GGUF copied to /kaggle/working/qwen35_finetuned_Q5_K_M.gguf")

shutil.make_archive("/kaggle/working/qwen35_lora", "zip", LORA_DIR)
print("LoRA zipped: /kaggle/working/qwen35_lora.zip")

# Other GGUF options (uncomment as needed):
# model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")
# model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="f16")
# model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method="q5_k_m", token="YOUR_HF_TOKEN")

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>